# 02 Fine-tune YOLO on IDD (Colab GPU)

The laptop has no NVIDIA GPU, so training happens here and only here. Everything upstream
(extract, verify, convert, EDA) is already done locally and ships as one tar file.

**Before running, from the laptop:**
1. `python scripts/train_yolo.py --dry-run` must pass (it does).
2. `git add -A && git commit -m "add training entry point" && git push` — the notebook clones
   the repo, so an unpushed `scripts/train_yolo.py` will not be there.
3. Upload `data/idd_yolo_640.tar` (2.4 GB) to `MyDrive/idd_p07/`. Upload the single tar, never
   the 41,794 loose files — Drive throttles badly on many small files.

**Session budget:** free Colab T4 gives ~4-12 h and disconnects after ~90 min idle. 30 epochs
on 31,569 images is roughly 2-3 h, so one session is normally enough. If it dies, the resume
cell picks up from the last epoch because checkpoints are written to Drive, not `/content`.

## 0. Confirm we actually got a GPU
Training on CPU here would take days, so fail fast.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import torch
print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())

## 1. Install Ultralytics
Colab images already carry a matching torch build, so only ultralytics is added.

In [ ]:
!pip -q install ultralytics==8.3.* 
import ultralytics
print('ultralytics', ultralytics.__version__)

## 2. Mount Drive
The tar and every checkpoint live here, so they survive the disconnect.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Paths and run parameters
All knobs in one place. `RUN_NAME` is what the summary file is called, so make it describe the run.

In [ ]:
# ---- configure these once, then run the notebook top to bottom ----
from pathlib import Path

DRIVE_ROOT = "/content/drive/MyDrive/idd_p07"          # your Drive folder
TAR        = f"{DRIVE_ROOT}/idd_yolo_640.tar"           # uploaded from the laptop
DATA_DIR   = "/content/idd_yolo"                        # ephemeral, wiped on disconnect
REPO_URL   = "https://github.com/ADHIINSVY13/P07_Indian_Road_Object_Detection.git"
BRANCH     = "feat/dataset-eda-pipeline"
REPO_DIR   = "/content/repo"

MODEL      = "yolo11n.pt"     # COCO-pretrained start point
EPOCHS     = 30
BATCH      = 16               # T4 16 GB handles 16 at 640 comfortably
IMGSZ      = 640
RUN_NAME   = "idd_yolo11n_e30"
PROJECT    = f"{DRIVE_ROOT}/runs"   # runs land on Drive, so a disconnect loses at most one epoch

print("tar   :", TAR, "exists:", Path(TAR).exists())
print("data  :", DATA_DIR)
print("run to:", f"{PROJECT}/{RUN_NAME}")

## 4. Unpack the dataset into local disk
`/content` is ephemeral and much faster than Drive. Re-running skips the work.

In [ ]:
import os, subprocess, time
if os.path.exists(f'{DATA_DIR}/images/train'):
    print('already extracted, skipping')
else:
    t = time.time()
    os.makedirs(DATA_DIR, exist_ok=True)
    subprocess.run(['tar', '-xf', TAR, '-C', DATA_DIR], check=True)
    print(f'extracted in {time.time() - t:.0f}s')

for split in ('train', 'val'):
    n = len(os.listdir(f'{DATA_DIR}/images/{split}'))
    m = len(os.listdir(f'{DATA_DIR}/labels/{split}'))
    print(f'{split}: {n} images, {m} labels', 'OK' if n == m else 'MISMATCH')
    assert n == m, f'{split}: image/label count mismatch -- do not train on this'

## 5. Get the code
`train_yolo.py --data-root` points the trainer at `/content/idd_yolo` instead of the repo-relative
`data/yolo`, and rewrites `idd.yaml` with this machine's paths, so the absolute path baked in on the
laptop is irrelevant.

In [ ]:
import os
if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} fetch --all --prune
else:
    !git clone {REPO_URL} {REPO_DIR}
!git -C {REPO_DIR} checkout {BRANCH}
!git -C {REPO_DIR} pull --ff-only
!git -C {REPO_DIR} log -1 --format='%h %ad %s' --date=short

## 6. Gate: the training entry point must exist and see the data
If this raises, the branch was not pushed from the laptop.

In [ ]:
%cd {REPO_DIR}
import os
assert os.path.exists('scripts/train_yolo.py'), 'scripts/train_yolo.py missing: commit and push it from the laptop first'
!python scripts/train_yolo.py --data-root {DATA_DIR} --model {MODEL} --epochs {EPOCHS} --batch {BATCH} --imgsz {IMGSZ} --name {RUN_NAME} --project {PROJECT} --dry-run

## 7. Train
`--project` points at Drive on purpose: `last.pt` is rewritten every epoch there, which is what
makes the resume cell work after a disconnect. Expect ~4-6 min per epoch after the first one
(the first also builds the label cache).

In [ ]:
%cd {REPO_DIR}
!python scripts/train_yolo.py --data-root {DATA_DIR} --model {MODEL} --epochs {EPOCHS} --batch {BATCH} --imgsz {IMGSZ} --workers 2 --name {RUN_NAME} --project {PROJECT}

## 8. Resume after a disconnect
Only run this if cell 7 was cut short. Ultralytics continues from the stored epoch count.

In [ ]:
%cd {REPO_DIR}
!python scripts/train_yolo.py --data-root {DATA_DIR} --model {MODEL} --epochs {EPOCHS} --batch {BATCH} --imgsz {IMGSZ} --workers 2 --name {RUN_NAME} --project {PROJECT} --resume

## 9. Results
`<RUN_NAME>_summary.json` holds per-class and per-group AP50-95. The group numbers are the
project's actual question: does IDD fine-tuning move the India-specific classes more than the
common ones? Ultralytics also writes `results.csv`, `confusion_matrix.png`, PR/F1 curves and
`weights/best.pt` into the run directory on Drive.

In [ ]:
import json
p = f'/content/repo/results/{RUN_NAME}_summary.json'
s = json.load(open(p))
print(json.dumps(s['metrics'], indent=2))
print(json.dumps(s['per_group_ap50_95'], indent=2))
print('weights:', s['weights_best'])

## 10. Keep the numbers in the repo
Download this small bundle and commit it — the report cites it, so it must not live only on Drive.

In [ ]:
import shutil, os
src = f'{PROJECT}/{RUN_NAME}'
bundle = f'{DRIVE_ROOT}/to_commit_{RUN_NAME}'
os.makedirs(bundle, exist_ok=True)
for f in ('results.csv', 'confusion_matrix.png', 'confusion_matrix_normalized.png', 'BoxPR_curve.png', 'BoxF1_curve.png'):
    if os.path.exists(f'{src}/{f}'):
        shutil.copy(f'{src}/{f}', bundle)
shutil.copy(f'/content/repo/results/{RUN_NAME}_summary.json', bundle)
print('collect these into the repo:', sorted(os.listdir(bundle)))